In [47]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd

import numpy as np
from IPython.display import clear_output

from python_utilities.db_connection import DbConnection


analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

INFO [2026-06-23 14:26:51] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [ ]:
start_date = "2026-05-01"
end_date = "2026-06-22"

query= f"""
SELECT la.*, tj.job_id, tj.status as textract_status, tj.s3_link 
FROM llm_attachments la LEFT JOIN textract_jobs tj ON la.attachment_id = tj.attachment_id
WHERE la.created_at >= '{start_date}' AND la.created_at <= '{end_date}'
"""


In [ ]:
data = analytics_db.sql_to_df(query)

In [ ]:
egvp_data = data[~data['attachment_id'].str.contains("-")]

In [ ]:
egvp_data.shape

In [ ]:
rejected = egvp_data[egvp_data["status"] == "rejected_too_long"].copy()
rejected.shape

In [ ]:
not_rejected_sample = egvp_data[egvp_data["status"] != "rejected_too_long"].sample(2000, random_state=42)
not_rejected_sample.shape

In [ ]:
egvp_sampled_one_month = pd.concat([rejected, not_rejected_sample], ignore_index=True)
egvp_sampled_one_month = egvp_sampled_one_month[egvp_sampled_one_month['textract_status']!= 'ERROR']

In [ ]:
egvp_sampled_one_month.textract_status.value_counts()

In [ ]:
egvp_sampled_one_month.columns

In [ ]:
import json

import boto3

session = boto3.Session(
    region_name="eu-central-1",
    profile_name="739275445236_DataScienceUser",
)
s3_client = session.client("s3")


def text_from_s3_link(s3_link: str) -> str:
    """Download a Textract-blocks JSON from S3 and merge LINE blocks into text."""
    bucket, key = s3_link.replace("s3://", "").split("/", 1)
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    blocks = json.loads(obj["Body"].read())
    if isinstance(blocks, dict):
        blocks = blocks.get("Blocks", [])
    return "\n".join(b["Text"] for b in blocks if b.get("BlockType") == "LINE")


In [ ]:
# s3_link points to a JSON of Textract blocks — fetch and merge LINE blocks
egvp_with_jobs = egvp_sampled_one_month[egvp_sampled_one_month["s3_link"].notna()].copy()

egvp_with_jobs["text"] = egvp_with_jobs["s3_link"].map(text_from_s3_link)
egvp_with_jobs[["attachment_id", "s3_link", "textract_status", "text"]].head()


In [ ]:
egvp_with_jobs

In [48]:

# egvp_with_jobs.to_csv("egvp_sampled_one_month_with_texts.csv", index=False)
egvp_sampled_one_month = pd.read_csv("egvp_sampled_one_month_with_texts.csv")

In [49]:
egvp_sampled_one_month

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,s3_key,s3_bucket,job_id,textract_status,s3_link,text
0,3519495,6ba656d1-8b2b-5e2a-ada9-3338261955b7,61608456,NaN,NaN,rejected_too_long,61608456_Dokumente_52426_30042026_084937_signe...,pdf,2026-05-01 09:22:55,2026-05-01 08:44:00,ocr_source_files/2026-05-01/egvp_id_359523/616...,pair-data-engineering-new,0df0a4dc09079f449f7358de7ed17c3cd40a5cae341db8...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Katharina Hahn\nHerrenstraße 11\nObergerichtsv...
1,3519729,45ff1f29-1ab0-56da-80fc-3ff54d56c88b,61609046,NaN,NaN,rejected_too_long,61609046_0_Sammel1.pdf,pdf,2026-05-01 11:13:46,2026-05-01 10:44:00,ocr_source_files/2026-05-01/egvp_id_359527/616...,pair-data-engineering-new,52dd8019502b010872a2b50b1c96a3f923b03f365def82...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieherin\nAmtsgericht: Burg\nM...
2,3519795,38d02160-d8c8-5f77-bc14-a519b9c6464a,61609291,NaN,NaN,rejected_too_long,61609291_Dokumente_42326_30042026_143651.pdf,pdf,2026-05-01 12:12:02,2026-05-01 11:44:00,ocr_source_files/2026-05-01/egvp_id_359530/616...,pair-data-engineering-new,e81857c58d61a105e970e7ad8ff5af509ef46c18f86b78...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Sandra Kaiser\nKirchstraße 9 15\nObergerichtsv...
3,3519799,06ccf869-98d4-5477-ba3b-3c03ab3d73c3,61609308,NaN,NaN,rejected_too_long,61609308_Dokumente_69426_01052026_121238.pdf,pdf,2026-05-01 12:12:02,2026-05-01 11:44:00,ocr_source_files/2026-05-01/egvp_id_359534/616...,pair-data-engineering-new,106a465a72538c8d7731f011ec3d15bcb4cb348b163e74...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,C. Heidenreich\nHändelstraße 4\nObergerichtsvo...
4,3519958,37ce1c0c-279b-505d-933d-20886223e6ac,61609719,NaN,NaN,rejected_too_long,61609719_DR_II_1284_26_Zustellung_erledigt.PDF,pdf,2026-05-01 14:11:58,2026-05-01 13:44:00,ocr_source_files/2026-05-01/egvp_id_359538/616...,pair-data-engineering-new,c22fe2def2e090b6dabdee01f68fd966a375077eeb585c...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieherin\nLeibnizstraße 102\nB...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3532,3553450,c1b49a3b-9a28-5933-8bdf-029c256953f7,62345629,NaN,NaN,processed,62345629_DRII-030926_BZSt_Ergebnis_1_08_05_202...,pdf,2026-05-08 13:17:50,2026-05-08 11:44:00,ocr_source_files/2026-05-08/egvp_id_365490/623...,pair-data-engineering-new,1f80f88bbaa3aeb21ee7bce5573f621559c88a3ccd9af4...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\n...
3533,3728662,3f513792-50d7-5142-ae07-62fd0cc92c07,68872957,NaN,NaN,processed,68872957_DR_II_1013_26_An_Gl_Mitt_Abgabe_Koll_...,pdf,2026-06-16 22:08:27,2026-06-16 20:44:00,ocr_source_files/2026-06-16/egvp_id_389810/688...,pair-data-engineering-new,15ebb740c9ef88af8c92993d22dfc0054f9e1f35f4c7ee...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Gerichtsvollzieherin (b)\nSprechstunden im Amt...
3534,3681441,4c1ed76c-524b-546f-a78b-6758a5c4b5ec,67882445,NaN,NaN,processed,67882445_DR-II_059626_Nachr_Mitlg__an_Glb_uebe...,pdf,2026-06-08 08:14:01,2026-06-08 06:44:00,ocr_source_files/2026-06-08/egvp_id_383294/678...,pair-data-engineering-new,454089d4de2f4631d9afa7422f3df1c689aece0ecefa80...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Bisdorfer Ring 16\n23769 Fehmarn\nTel. 0157/36...
3535,3657871,1eb3645a-698a-5c45-a815-384ab5915e9c,67122224,NaN,NaN,processed,67122224_WiderspruchsN_2610270260006E_926c8273...,pdf,2026-06-02 10:58:21,2026-06-02 09:44:00,ocr_source_files/2026-06-02/egvp_id_380369/671...,pair-data-engineering-new,718db15289ca4413b8dcc79f4e5ed7657ae57d184c36a0...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Wedding\nMahnsache Liquandum Capit...


In [51]:
egvp_sampled_one_month.status.value_counts()

status
processed             1998
rejected_too_long     1538
rejected_too_short       1
Name: count, dtype: int64

In [52]:
check = egvp_sampled_one_month[egvp_sampled_one_month["status"] == "rejected_too_short"].iloc[0]
check

id                                                             3655090
ticket_uuid                       f143af44-2ca1-54d7-980c-3ae77e52d4df
attachment_id                                                 66998662
zendesk_id                                                         NaN
comment_id                                                         NaN
status                                              rejected_too_short
file_name                  66998662_Dokument_17926_01062026_195028.pdf
file_extension                                                     pdf
status_written_at                                  2026-06-01 19:23:26
created_at                                         2026-06-01 18:44:00
s3_key               ocr_source_files/2026-06-01/egvp_id_380088/669...
s3_bucket                                    pair-data-engineering-new
job_id               eb756dadc7750628a2ae9db0a552ae656cb346e98a3bef...
textract_status                                              SUCCEEDED
s3_lin

In [53]:
egvp_sampled_one_month.columns

Index(['id', 'ticket_uuid', 'attachment_id', 'zendesk_id', 'comment_id',
       'status', 'file_name', 'file_extension', 'status_written_at',
       'created_at', 's3_key', 's3_bucket', 'job_id', 'textract_status',
       's3_link', 'text'],
      dtype='object')

In [54]:
egvp_sampled_one_month["text"].isna().sum()

np.int64(1)

In [55]:
# drop na text
egvp_sampled_one_month = egvp_sampled_one_month[egvp_sampled_one_month["text"].notna()].copy()

In [56]:
max_attachment_length_tokens_egvp = 1500

get_length = lambda text: len(text.strip().split()) if text else 0

egvp_sampled_one_month["simple_token_count"] = egvp_sampled_one_month["text"].map(get_length)


In [57]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

tokens = egvp_sampled_one_month["simple_token_count"]
limit = max_attachment_length_tokens_egvp

# CDF data
sorted_tokens = np.sort(tokens)
cdf = np.arange(1, len(sorted_tokens) + 1) / len(sorted_tokens)
pct_below = (tokens <= limit).mean() * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Simple Token Count Distribution (log y-scale)",
        "Cumulative Distribution of Simple Token Counts",
    ),
)

# Left: histogram with log y-scale
fig.add_trace(
    go.Histogram(
        x=tokens,
        nbinsx=50,
        marker=dict(color="skyblue", line=dict(color="black", width=1)),
        name="Token count",
        hovertemplate="Tokens: %{x}<br>Count: %{y}<extra></extra>",
    ),
    row=1, col=1,
)

# Right: CDF
fig.add_trace(
    go.Scatter(
        x=sorted_tokens,
        y=cdf,
        mode="lines",
        line=dict(color="steelblue", width=2),
        name="CDF",
        hovertemplate="Tokens ≤ %{x}<br>Fraction: %{y:.2%}<extra></extra>",
    ),
    row=1, col=2,
)

# Limit line on both subplots
for col in (1, 2):
    fig.add_vline(
        x=limit, line=dict(color="red", dash="dash", width=2),
        annotation_text=f"Limit ({limit})", annotation_position="top",
        row=1, col=col,
    )

# % below limit guide line on the CDF
fig.add_hline(
    y=pct_below / 100, line=dict(color="orange", dash="dot", width=1.5),
    annotation_text=f"{pct_below:.1f}% below limit", annotation_position="bottom right",
    row=1, col=2,
)

fig.update_yaxes(type="log", title_text="Frequency (log)", row=1, col=1)
fig.update_xaxes(title_text="Token Count", row=1, col=1)
fig.update_yaxes(title_text="Cumulative Fraction", row=1, col=2)
fig.update_xaxes(title_text="Token Count", row=1, col=2)

fig.update_layout(
    title_text="EGVP Attachments — Simple Token Count Analysis",
    showlegend=False,
    template="plotly_white",
    height=500, width=1100,
    bargap=0.05,
)

fig.show()

print(tokens.describe().round())
print(f"\n% above limit: {(tokens > limit).mean()*100:.1f}%")

count     3536.0
mean      1554.0
std       2235.0
min         33.0
25%        276.0
50%        658.0
75%       2509.0
max      68375.0
Name: simple_token_count, dtype: float64

% above limit: 43.5%


In [58]:
egvp_sampled_one_month["simple_token_count"].mean()

np.float64(1553.8526583710407)

In [59]:
egvp_sampled_one_month["simple_token_count"].median()

np.float64(658.0)

# Tokenize with qwen3 tokenizer to see real tokenized dist

In [60]:
from transformers import AutoTokenizer
tokenizer_32B = AutoTokenizer.from_pretrained("Qwen/QwQ-32B")

In [61]:
egvp_sampled_one_month['real_token_count'] = egvp_sampled_one_month['text'].map(lambda t: len(tokenizer_32B.encode(t, add_special_tokens=False)))

Token indices sequence length is longer than the specified maximum sequence length for this model (162169 > 131072). Running this sequence through the model will result in indexing errors


In [62]:
egvp_sampled_one_month['real_token_count'].describe()

count      3536.000000
mean       4651.942308
std        8598.864389
min         130.000000
25%         840.000000
50%        2153.000000
75%        7035.500000
max      281631.000000
Name: real_token_count, dtype: float64

In [63]:
np.median(egvp_sampled_one_month['real_token_count'])

np.float64(2153.0)

In [64]:
egvp_sampled_one_month['simple_token_count'].describe()

count     3536.000000
mean      1553.852658
std       2235.049825
min         33.000000
25%        276.000000
50%        658.000000
75%       2509.000000
max      68375.000000
Name: simple_token_count, dtype: float64

In [65]:
np.median(egvp_sampled_one_month['simple_token_count'])

np.float64(658.0)

In [66]:
import plotly.graph_objects as go

cmp = egvp_sampled_one_month[["simple_token_count", "real_token_count"]].dropna()

# Per-document ratio of real (Qwen) tokens to simple whitespace tokens
cmp = cmp.assign(ratio=cmp["real_token_count"] / cmp["simple_token_count"].replace(0, np.nan))

print("Summary:")
print(cmp[["simple_token_count", "real_token_count", "ratio"]].describe().round(2))
print(f"\nPearson correlation: {cmp['simple_token_count'].corr(cmp['real_token_count']):.4f}")
print(f"Median real/simple ratio: {cmp['ratio'].median():.2f}")
print(f"Mean real/simple ratio:   {cmp['ratio'].mean():.2f}")

# Scatter: simple vs real token count, with y=x reference line
max_val = int(cmp[["simple_token_count", "real_token_count"]].to_numpy().max())

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cmp["simple_token_count"], y=cmp["real_token_count"],
    mode="markers",
    marker=dict(color="steelblue", size=5, opacity=0.5),
    name="documents",
    hovertemplate="simple: %{x}<br>real: %{y}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode="lines", line=dict(color="red", dash="dash"),
    name="y = x",
))
fig.update_layout(
    title_text="Real (Qwen) vs Simple (whitespace) Token Count",
    xaxis_title="simple_token_count",
    yaxis_title="real_token_count",
    template="plotly_white",
    height=550, width=650,
)
fig.show()


Summary:
       simple_token_count  real_token_count    ratio
count             3536.00           3536.00  3536.00
mean              1553.85           4651.94     3.06
std               2235.05           8598.86     0.42
min                 33.00            130.00     2.15
25%                276.00            840.00     2.80
50%                658.00           2153.00     2.94
75%               2509.00           7035.50     3.21
max              68375.00         281631.00     7.00

Pearson correlation: 0.9388
Median real/simple ratio: 2.94
Mean real/simple ratio:   3.06


In [67]:
rejected_egvp = egvp_sampled_one_month[egvp_sampled_one_month["status"] == "rejected_too_long"].copy()


In [68]:
np.mean(rejected_egvp['real_token_count'] / rejected_egvp['simple_token_count'])

np.float64(2.8965086047433064)

In [69]:
np.median(rejected_egvp['real_token_count'] / rejected_egvp['simple_token_count'])

np.float64(2.8217954207241043)

# real token count cdm plots

In [70]:
import numpy as np
import plotly.graph_objects as go

def _cdf(values):
    v = np.sort(np.asarray(values, dtype=float))
    y = np.arange(1, len(v) + 1) / len(v)
    return v, y

all_tokens = egvp_sampled_one_month["real_token_count"].dropna()

x_all, y_all = _cdf(all_tokens)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_all, y=y_all, mode="lines", name=f"All (n={len(all_tokens)})",
    line=dict(color="steelblue", width=2),
    hovertemplate="real tokens ≤ %{x}<br>fraction: %{y:.2%}<extra></extra>",
))

limit_adjusted = max_attachment_length_tokens_egvp * np.median(rejected_egvp['real_token_count'] / rejected_egvp['simple_token_count'])

fig.add_vline(
    x=limit_adjusted,
    line=dict(color="red", dash="dash", width=2),
    annotation_text=f"Adjusted Limit ({limit_adjusted:.0f})",
    annotation_position="top",
)

fig.update_layout(
    title_text="CDF of Real (Qwen) Token Count — All Data",
    xaxis_title="real_token_count",
    yaxis_title="Cumulative Fraction",
    template="plotly_white",
    height=550, width=800,
)
fig.show()

print("All data real_token_count:")
print(all_tokens.describe().round())


All data real_token_count:
count      3536.0
mean       4652.0
std        8599.0
min         130.0
25%         840.0
50%        2153.0
75%        7036.0
max      281631.0
Name: real_token_count, dtype: float64


In [71]:
import plotly.graph_objects as go

x_max = 20000
counts_in_range = (egvp_sampled_one_month["real_token_count"] <= x_max).sum()
counts_above = (egvp_sampled_one_month["real_token_count"] > x_max).sum()

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=egvp_sampled_one_month["real_token_count"].clip(upper=x_max),
    xbins=dict(start=0, end=x_max, size=200),
    marker=dict(color="skyblue", line=dict(color="black", width=0.5)),
    name="Real token count",
    hovertemplate="real tokens: %{x}<br>Count: %{y}<extra></extra>",
))

fig.add_vline(
    x=limit_adjusted,
    line=dict(color="red", dash="dash", width=2),
    annotation_text=f"Real Token Count Limit App ({limit_adjusted.round(2)})",
    annotation_position="top right",
)

fig.update_layout(
    title_text=f"Histogram of Real (Qwen) Token Count (0–{x_max:,}) — {counts_above} docs clipped above {x_max:,}",
    xaxis_title="real_token_count",
    yaxis_title="Count",
    template="plotly_white",
    height=500, width=1000,
    bargap=0.02,
)
fig.update_xaxes(range=[0, x_max], dtick=1000, tickangle=45)
fig.show()

print(f"Docs ≤ {x_max:,}: {counts_in_range}  |  Docs > {x_max:,}: {counts_above}")


Docs ≤ 20,000: 3518  |  Docs > 20,000: 18


In [72]:
egvp_sampled_one_month.status.value_counts()

status
processed            1998
rejected_too_long    1538
Name: count, dtype: int64

In [73]:
egvp_sampled_one_month.shape

(3536, 18)

In [74]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x_max = 20000
bin_size = 200

processed_egvp = egvp_sampled_one_month[egvp_sampled_one_month["status"] != "rejected_too_long"].copy()

groups = {
    "Rejected (too long)": rejected_egvp["real_token_count"],
    "Processed": processed_egvp["real_token_count"],
}
colors = {"Rejected (too long)": "salmon", "Processed": "skyblue"}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Rejected too long (n={len(groups['Rejected (too long)'])})",
        f"Processed (n={len(groups['Processed'])})",
    ],
    shared_yaxes=False,
)

for col_idx, (label, series) in enumerate(groups.items(), start=1):
    clipped = series.dropna().clip(upper=x_max)
    fig.add_trace(go.Histogram(
        x=clipped,
        xbins=dict(start=0, end=x_max, size=bin_size),
        marker=dict(color=colors[label], line=dict(color="black", width=0.5)),
        name=label,
        hovertemplate="real tokens: %{x}<br>Count: %{y}<extra></extra>",
        showlegend=False,
    ), row=1, col=col_idx)

    fig.add_vline(
        x=limit_adjusted,
        line=dict(color="red", dash="dash", width=2),
        annotation_text=f"Adjusted Limit ({limit_adjusted:.0f})",
        annotation_position="top right",
        row=1, col=col_idx,
    )

    fig.update_xaxes(range=[0, x_max], dtick=2000, tickangle=45, title_text="real_token_count", row=1, col=col_idx)
    fig.update_yaxes(title_text="Count", row=1, col=col_idx)

fig.update_layout(
    title_text="Real (Qwen) Token Count — Rejected vs Processed (0–20k)",
    template="plotly_white",
    height=500, width=1100,
    bargap=0.02,
)
fig.show()

for label, series in groups.items():
    print(f"\n{label}:")
    print(series.dropna().describe().round())
    print(f"  > {x_max:,}: {(series.dropna() > x_max).sum()} docs")



Rejected (too long):
count      1538.0
mean       9145.0
std       11550.0
min        4070.0
25%        5938.0
50%        7401.0
75%       10011.0
max      281631.0
Name: real_token_count, dtype: float64
  > 20,000: 18 docs

Processed:
count    1998.0
mean     1193.0
std       835.0
min       130.0
25%       708.0
50%       860.0
75%      1305.0
max      6377.0
Name: real_token_count, dtype: float64
  > 20,000: 0 docs


In [75]:
np.median(processed_egvp["real_token_count"])

np.float64(860.5)

In [76]:
np.median(rejected_egvp["real_token_count"])

np.float64(7401.0)

In [77]:
import numpy as np
import plotly.graph_objects as go

x_max = 20000

def _cdf(values, clip=None):
    v = np.sort(np.asarray(values.dropna(), dtype=float))
    if clip is not None:
        v = np.clip(v, None, clip)
    y = np.arange(1, len(v) + 1) / len(v)
    return v, y

groups = {
    "All": (egvp_sampled_one_month["real_token_count"], "steelblue"),
    "Rejected (too long)": (rejected_egvp["real_token_count"], "firebrick"),
    "Processed": (processed_egvp["real_token_count"], "seagreen"),
}

fig = go.Figure()
for label, (series, color) in groups.items():
    x, y = _cdf(series, clip=x_max)
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="lines",
        name=f"{label} (n={len(series.dropna())})",
        line=dict(color=color, width=2),
        hovertemplate=f"{label}<br>real tokens ≤ %{{x}}<br>fraction: %{{y:.2%}}<extra></extra>",
    ))

fig.add_vline(
    x=limit_adjusted,
    line=dict(color="red", dash="dash", width=2),
    annotation_text=f"Adjusted Limit ({limit_adjusted:.0f})",
    annotation_position="top left",
)

fig.update_layout(
    title_text=f"CDF of Real (Qwen) Token Count — All / Rejected / Processed (clipped at {x_max:,})",
    xaxis_title="real_token_count",
    yaxis_title="Cumulative Fraction",
    template="plotly_white",
    height=550, width=900,
    xaxis=dict(range=[0, x_max], dtick=2000),
)
fig.show()

for label, (series, _) in groups.items():
    pct_below = (series.dropna() <= limit_adjusted).mean() * 100
    print(f"{label}: {pct_below:.1f}% below adjusted limit ({limit_adjusted:.0f})")


All: 56.2% below adjusted limit (4233)
Rejected (too long): 0.5% below adjusted limit (4233)
Processed: 99.0% below adjusted limit (4233)


In [78]:
v,y = _cdf(rejected_egvp["real_token_count"].dropna())

In [79]:
len(v)

1538

In [80]:
# check token limit range 4k-7k
check_range = (v >= 4000) & (v <= 10000)
# get every 10th value in this range
v_range_fractioned = v[check_range][::25]
y_range_fractioned = y[check_range][::25] * 100


In [81]:
# plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=v_range_fractioned, y=y_range_fractioned,
    mode="markers+lines",
    line=dict(color="steelblue", width=2),
    marker=dict(color="steelblue", size=6),
    name="rejected too long",
    hovertemplate="real tokens ≤ %{x}<br>fraction: %{y:.2f}%<extra></extra>",
))
fig.add_vline(
    x=limit_adjusted,
    line=dict(color="red", dash="dash", width=2),
    annotation_text=f"Adjusted Limit ({limit_adjusted:.0f})",
    annotation_position="top left",
)
fig.update_layout(
    title_text="CDF of Real Token Count for Rejected EGVP (4k–10k range, every 25th point)",
    xaxis_title="real_token_count",
    yaxis_title="Cumulative Percentage",
    template="plotly_white",
    height=500, width=700,
)
fig.show()

In [82]:
len(v[check_range])

1152

# check our prompts and see how many tokens they actually have

In [83]:
prompt_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/intent_recognition/configs/models"


model_to_prompt = {}
for json_file in os.listdir(prompt_dir):
    if json_file.endswith(".json"):
        with open(os.path.join(prompt_dir, json_file), "r") as f:
            config = json.load(f)
            name = config.get("name", "unknown")
            model_path = config.get("model", {}).get("model_path", "unknown")
            preprocessing = config.get("preprocessing", {})
            prompt_config = preprocessing.get('prompt', {})
            if "template" in prompt_config:
                template = prompt_config["template"]
                # prompt is either under prompt or user key
                prompt_text = template.get("prompt") or template.get("user")
                if prompt_text and len(prompt_text) > 100:
                    print(f"Config: {name}, Model: {model_path}, Prompt length: {len(prompt_text)}")
                    model_to_prompt[name] = {"prompt": prompt_text, "model_path": model_path}
            

Config: egvp_standalone_invoice, Model: models/QwQ-32B, Prompt length: 6372
Config: dispute_insolvency, Model: models/Meta-Llama-3.1-8B-Instruct, Prompt length: 2979
Config: creditor_communication, Model: models/Qwen3-4B, Prompt length: 5293
Config: gdpr_high_precision_v3, Model: models/gpt-oss-20b, Prompt length: 11543
Config: ip_and_iprev, Model: models/qwen3-4b-ip_and_iprev_reasoning_experiment_v2_checkpoint-4000, Prompt length: 10349
Config: drittauskunft_egvp, Model: models/Qwen3-4B, Prompt length: 8208
Config: gdpr, Model: models/GDPR_request_llama3_8b_instruct_merged, Prompt length: 1198
Config: cannot_pay_combo_2, Model: models/cannot_pay_combo_2, Prompt length: 10493
Config: pfub_erlass_egvp, Model: models/QwQ-32B, Prompt length: 5243
Config: vermogenverzeichnis_egvp, Model: models/Qwen3-4B, Prompt length: 5102
Config: dispute_contract_cancelled, Model: models/disputes_ccenter_finetuned, Prompt length: 523
Config: debt_counseling, Model: models/Meta-Llama-3-8B-Instruct, Prompt

In [84]:
# calculate token counts for each prompt

model_to_token_count = {
    model: {
        "model_path": info["model_path"],
        "token_count": len(tokenizer_32B.encode(info["prompt"], add_special_tokens=True)),
    }
    for model, info in model_to_prompt.items()
}
model_to_token_count

{'egvp_standalone_invoice': {'model_path': 'models/QwQ-32B',
  'token_count': 1626},
 'dispute_insolvency': {'model_path': 'models/Meta-Llama-3.1-8B-Instruct',
  'token_count': 636},
 'creditor_communication': {'model_path': 'models/Qwen3-4B',
  'token_count': 1218},
 'gdpr_high_precision_v3': {'model_path': 'models/gpt-oss-20b',
  'token_count': 2512},
 'ip_and_iprev': {'model_path': 'models/qwen3-4b-ip_and_iprev_reasoning_experiment_v2_checkpoint-4000',
  'token_count': 2070},
 'drittauskunft_egvp': {'model_path': 'models/Qwen3-4B', 'token_count': 2300},
 'gdpr': {'model_path': 'models/GDPR_request_llama3_8b_instruct_merged',
  'token_count': 276},
 'cannot_pay_combo_2': {'model_path': 'models/cannot_pay_combo_2',
  'token_count': 2496},
 'pfub_erlass_egvp': {'model_path': 'models/QwQ-32B', 'token_count': 1450},
 'vermogenverzeichnis_egvp': {'model_path': 'models/Qwen3-4B',
  'token_count': 1383},
 'dispute_contract_cancelled': {'model_path': 'models/disputes_ccenter_finetuned',
  't

In [85]:
# plot model_to_token_count as bar chart
fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(model_to_token_count.keys()),
    y=[v["token_count"] for v in model_to_token_count.values()],
    text=[v["model_path"] for v in model_to_token_count.values()],
    textposition="outside",
    marker=dict(color="skyblue", line=dict(color="black", width=1)),
))
fig.update_layout(
    title_text="Token Counts for Model Prompts",
    xaxis_title="Model",
    yaxis_title="Token Count",
    template="plotly_white",
    height=500, width=800,

)

fig.show()

In [86]:
# egvp specific prompts

egvp_only = [
    "pfub_erlass_egvp",
    "invoice_detection_egvp",
    "vermogenverzeichnis_egvp",
    "egvp_standalone_invoice",
    "drittauskunft_egvp",
]

In [87]:
model_to_token_count_egvp = {model: info for model, info in model_to_token_count.items() if model in egvp_only}

mean_egvp_tokens = np.mean([v["token_count"] for v in model_to_token_count_egvp.values()])

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(model_to_token_count_egvp.keys()),
    y=[v["token_count"] for v in model_to_token_count_egvp.values()],
    text=[v["token_count"] for v in model_to_token_count_egvp.values()],
    textposition="outside",
    marker=dict(color="salmon", line=dict(color="black", width=1)),
    hovertemplate="Model: %{x}<br>Tokens: %{y}<br>LLM: %{customdata}<extra></extra>",
    customdata=[v["model_path"] for v in model_to_token_count_egvp.values()],
))
fig.add_hline(
    y=mean_egvp_tokens,
    line=dict(color="red", dash="dash", width=2),
    annotation_text=f"Mean ({mean_egvp_tokens:.0f})",
    annotation_position="top left",
)
fig.update_layout(
    title_text="Token Counts for EGVP-specific Model Prompts",
    xaxis_title="Model",
    yaxis_title="Token Count",
    template="plotly_white",
    height=500, width=800,
    margin=dict(l=40, r=20, t=50, b=40),
)
fig.show()

In [88]:
mean_egvp_tokens = np.mean([v["token_count"] for model, v in model_to_token_count.items() if model in egvp_only])
mean_egvp_tokens

np.float64(1885.2)

In [89]:
# token-length related generation params for each egvp model
# note: "max_lenght" is the (misspelled) key as it appears in the json
token_len_keys = ["max_lenght", "max_new_tokens"]

for json_file in os.listdir(prompt_dir):
    if not json_file.endswith(".json"):
        continue
    with open(os.path.join(prompt_dir, json_file), "r") as f:
        config = json.load(f)
    name = config.get("name", "unknown")
    if name not in egvp_only:
        continue
    gen_params = config.get("generation_params", {})
    token_params = {k: gen_params.get(k) for k in token_len_keys}
    print(f"{name} ({config.get('model', {}).get('model_path', 'unknown')}): {token_params}")


egvp_standalone_invoice (models/QwQ-32B): {'max_lenght': None, 'max_new_tokens': 2012}
drittauskunft_egvp (models/Qwen3-4B): {'max_lenght': 8000, 'max_new_tokens': 5000}
pfub_erlass_egvp (models/QwQ-32B): {'max_lenght': None, 'max_new_tokens': 2000}
vermogenverzeichnis_egvp (models/Qwen3-4B): {'max_lenght': 8000, 'max_new_tokens': 5000}
invoice_detection_egvp (models/QwQ-32B): {'max_lenght': None, 'max_new_tokens': 2000}


In [90]:
# Effect of the text-length limit on real (Qwen) token counts of ACCEPTED egvp data.
# The limit is applied on simple_token_count (whitespace word count): a document is
# "accepted" when simple_token_count <= limit. For each limit we look at the real
# token-count distribution of the accepted subset.

limit_sweep = range(1500, 2501, 50)

limit_stats = []
for lim in limit_sweep:
    accepted = egvp_sampled_one_month[egvp_sampled_one_month["simple_token_count"] <= lim]
    real_tokens = accepted["real_token_count"].dropna()
    limit_stats.append({
        "limit": lim,
        "n_accepted": len(real_tokens),
        "mean": real_tokens.mean(),
        "median": real_tokens.median(),
        "std": real_tokens.std(),
        "min": real_tokens.min(),
        "max": real_tokens.max(),
        "p90": real_tokens.quantile(0.90),
        "p95": real_tokens.quantile(0.95),
        "p99": real_tokens.quantile(0.99),
    })

limit_stats_df = pd.DataFrame(limit_stats)
limit_stats_df.round(1)


,limit,n_accepted,mean,median,std,min,max,p90,p95,p99
0,1500,1999,1194.9,861.0,837.8,130,6377,2340.2,3113.1,4218.0
1,1550,2056,1289.0,874.0,1000.3,130,6965,2715.5,3888.0,4691.8
2,1600,2093,1351.2,883.0,1097.0,130,6965,2959.2,4254.6,5038.4
3,1650,2132,1418.7,894.5,1197.2,130,6965,3377.3,4397.7,5343.1
4,1700,2155,1457.5,902.0,1250.1,130,7137,3740.0,4530.0,5399.3
5,1750,2176,1492.9,905.0,1295.8,130,7137,3922.0,4646.2,5472.0
6,1800,2206,1546.5,908.0,1368.3,130,8824,4221.0,4793.0,5700.2
7,1850,2238,1604.4,913.0,1444.4,130,9259,4344.0,4975.2,5935.7
8,1900,2264,1651.3,918.5,1503.4,130,9259,4466.5,5136.3,6026.6
9,1950,2287,1691.4,925.0,1548.1,130,9259,4607.6,5246.1,6127.9


In [91]:
# Visualize the effect of the limit on real token-count statistics of accepted data.
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Real Token Count Stats vs Text-Length Limit",
        "Number of Accepted Docs vs Text-Length Limit",
    ),
)

stat_lines = {
    "mean": "steelblue",
    "median": "seagreen",
    "p90": "orange",
    "p95": "salmon",
    "p99": "firebrick",
    "max": "purple",
}
for stat, color in stat_lines.items():
    fig.add_trace(go.Scatter(
        x=limit_stats_df["limit"], y=limit_stats_df[stat],
        mode="lines+markers", name=stat,
        line=dict(color=color, width=2), marker=dict(size=5),
        hovertemplate=f"limit: %{{x}}<br>{stat} real tokens: %{{y:.0f}}<extra></extra>",
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=limit_stats_df["limit"], y=limit_stats_df["n_accepted"],
    mode="lines+markers", name="n_accepted",
    line=dict(color="black", width=2), marker=dict(size=5),
    hovertemplate="limit: %{x}<br>accepted docs: %{y}<extra></extra>",
), row=1, col=2)

fig.update_xaxes(title_text="Text-length limit (simple_token_count)", dtick=100, tickangle=45, row=1, col=1)
fig.update_xaxes(title_text="Text-length limit (simple_token_count)", dtick=100, tickangle=45, row=1, col=2)
fig.update_yaxes(title_text="Real token count", row=1, col=1)
fig.update_yaxes(title_text="Accepted docs", row=1, col=2)

fig.update_layout(
    title_text="Effect of Text-Length Limit (1500–2500) on Accepted EGVP Real Token Counts",
    template="plotly_white",
    height=500, width=1100,
)
fig.show()


# Lets change on avarage how much token do we need to answer the question?

In [164]:
egvp_sampled_one_month.shape

(3536, 18)

In [94]:
egvp_processed = egvp_sampled_one_month[egvp_sampled_one_month["status"] == "processed"].copy()
egvp_processed.shape

(1998, 18)

In [95]:
egvp_processed

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,s3_key,s3_bucket,job_id,textract_status,s3_link,text,simple_token_count,real_token_count
1538,3561360,d3c6ad76-190a-5c04-b1df-fa03eecd25e6,62544048,NaN,NaN,processed,62544048_Vermoegensverzeichnis.pdf,pdf,2026-05-11 11:53:43,2026-05-11 10:44:00,ocr_source_files/2026-05-11/egvp_id_366588/625...,pair-data-engineering-new,65b68b97de6393ab535f92100a098c3ad451b90df6e798...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d. Obergerichtsvollzi...,1322,3841
1539,3604600,5e011188-012f-57c2-a420-ef052cd6747e,65187685,NaN,NaN,processed,65187685_Dokument_74626_20052026_110542.pdf,pdf,2026-05-20 15:25:00,2026-05-20 12:44:00,ocr_source_files/2026-05-20/egvp_id_373558/651...,pair-data-engineering-new,117caa13764cf950af4bd423a065ab854399dffd06551b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Andreas Voß\nObergerichtsvollzieher\nNeustraße...,175,550
1540,3631112,0f2f8652-8d05-5f47-aa6f-06dabae87fd1,65441945,NaN,NaN,processed,65441945_Dokument_36026_27052026_104646.pdf,pdf,2026-05-27 12:39:35,2026-05-27 11:44:00,ocr_source_files/2026-05-27/egvp_id_377707/654...,pair-data-engineering-new,54861736df6de969ba850a9a7984bbbb1afc1f83c9d516...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Inga Rack\nHellersdorfer Weg 35\nObergerichtsv...,196,666
1541,3604598,4552df9f-70a1-5bf8-8729-9692ee297faa,65187678,NaN,NaN,processed,65187678_Dokument_74026_20052026_103307.pdf,pdf,2026-05-20 15:24:59,2026-05-20 12:44:00,ocr_source_files/2026-05-20/egvp_id_373556/651...,pair-data-engineering-new,e55c3622245ee9b2714c7ed069ed6403d3cc78e5e7b1e1...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,M. Hommel\nBrunsbütteler Damm 446\nGerichtsvol...,236,763
1542,3727126,941dd9ce-d62b-5782-a848-62c115a48b03,68869114,NaN,NaN,processed,68869114_DR_II_1159_26_aus_Schreibmaschine.PDF,pdf,2026-06-16 15:24:01,2026-06-16 13:44:00,ocr_source_files/2026-06-16/egvp_id_389615/688...,pair-data-engineering-new,1ddd52b73afb8e205e19f67416c292354d4f7d20638ec9...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Gerichtsvollzieher\nAmtsgericht\nStuttgart\nSc...,262,746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3532,3553450,c1b49a3b-9a28-5933-8bdf-029c256953f7,62345629,NaN,NaN,processed,62345629_DRII-030926_BZSt_Ergebnis_1_08_05_202...,pdf,2026-05-08 13:17:50,2026-05-08 11:44:00,ocr_source_files/2026-05-08/egvp_id_365490/623...,pair-data-engineering-new,1f80f88bbaa3aeb21ee7bce5573f621559c88a3ccd9af4...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\n...,878,3180
3533,3728662,3f513792-50d7-5142-ae07-62fd0cc92c07,68872957,NaN,NaN,processed,68872957_DR_II_1013_26_An_Gl_Mitt_Abgabe_Koll_...,pdf,2026-06-16 22:08:27,2026-06-16 20:44:00,ocr_source_files/2026-06-16/egvp_id_389810/688...,pair-data-engineering-new,15ebb740c9ef88af8c92993d22dfc0054f9e1f35f4c7ee...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Gerichtsvollzieherin (b)\nSprechstunden im Amt...,363,1057
3534,3681441,4c1ed76c-524b-546f-a78b-6758a5c4b5ec,67882445,NaN,NaN,processed,67882445_DR-II_059626_Nachr_Mitlg__an_Glb_uebe...,pdf,2026-06-08 08:14:01,2026-06-08 06:44:00,ocr_source_files/2026-06-08/egvp_id_383294/678...,pair-data-engineering-new,454089d4de2f4631d9afa7422f3df1c689aece0ecefa80...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Bisdorfer Ring 16\n23769 Fehmarn\nTel. 0157/36...,178,601
3535,3657871,1eb3645a-698a-5c45-a815-384ab5915e9c,67122224,NaN,NaN,processed,67122224_WiderspruchsN_2610270260006E_926c8273...,pdf,2026-06-02 10:58:21,2026-06-02 09:44:00,ocr_source_files/2026-06-02/egvp_id_380369/671...,pair-data-engineering-new,718db15289ca4413b8dcc79f4e5ed7657ae57d184c36a0...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Wedding\nMahnsache Liquandum Capit...,307,835


# Fetch gpu_tasks data for egvp_processed ticket_uuids
The `gpu_tasks` table lives in the GRAPH DB (`[GRAPH_DB]` / `GRAPH_DATABASE_URI` in secret.ini).

In [96]:
graph_db = DbConnection('GRAPH_DB', 'GRAPH_DATABASE_URI')

ticket_uuids = egvp_processed['ticket_uuid'].dropna().unique().tolist()
print(f"Querying gpu_tasks for {len(ticket_uuids)} ticket_uuids")

gpu_tasks_query = "SELECT * FROM gpu_tasks WHERE ticket_uuid IN %(ticket_uuids)s"
gpu_tasks = graph_db.sql_to_df(gpu_tasks_query, params={"ticket_uuids": ticket_uuids})
gpu_tasks.shape

INFO [2026-06-23 14:39:26] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


Querying gpu_tasks for 1948 ticket_uuids


(13882, 13)

In [97]:
gpu_tasks.head()

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,created_at,updated_at,attachment_id,meta_info
0,55c71e28-b4f4-4cd5-a5f9-8229ac70fc44,9267c7e9-706d-46ee-a3f9-e0d45a13b5e3,d99f9de4-e53b-5bde-ba45-25be1ddad3e9,aftercourt_classification_vermogenverzeichnis,done,{'text': 'Anwaltskanzlei Hövel & Collegen Raim...,"{'class_pred': False, 'class_prob': 0.02, 'aft...","{'class_pred': False, 'class_prob': 0.02, 'aft...",0,2026-05-29 12:32:19.216128,2026-05-29 12:32:19.216128,65623690,None
1,3c90f072-1897-4022-95f9-0af8ca53e9c3,0b1b13c8-5492-41f9-acd3-8575ad3ea1e7,b9904f6b-a18d-5d5a-a2b9-8dec0691918a,aftercourt_classification_ladung,done,{'text': 'Amtsgericht Schleswig Mahnsache free...,"{'class_pred': False, 'class_prob': 0.04, 'aft...","{'class_pred': False, 'class_prob': 0.04, 'aft...",0,2026-06-10 14:29:01.480465,2026-06-10 14:29:01.480465,68264187,None
2,55bd7cf1-ef44-4c22-83ad-fe8ec4fae23b,76afcc13-ab74-43b6-810c-566cbe1e3725,ad0a5cc5-c62b-5c79-a8f6-ea0b25401e39,aftercourt_classification_ladung,done,{'text': 'Würzburger Str. 14 (Haus A) 01187 Dr...,"{'class_pred': False, 'class_prob': 0.01, 'aft...","{'class_pred': False, 'class_prob': 0.01, 'aft...",0,2026-05-26 12:24:53.715100,2026-05-26 12:24:53.715100,65405983,None
3,f8cdac37-b141-4f0f-ac57-7b255db3c801,3b7fe2c3-62d5-4ff1-8de3-f3c5955c543a,6d4075a2-7bde-5f60-88ac-ea0aed86c059,invoice_detection_egvp,done,{'text': '<page_1> Beleg über die Einlieferung...,"{'answer': '{""is_invoice_inside"": true, ""start...","{'end_page': 1, 'start_page': 1, 'is_invoice_i...",0,2026-05-29 12:29:13.651323,2026-05-29 12:53:31.070652,65612554,"{'instance': {'private_ip': '100.2.38.122', 'i..."
4,378b33de-ead5-42c3-b099-515bba2cc6a4,da1ad76a-a880-4a2d-8b3b-261106498a34,011af6c2-a852-5e94-8039-bdada6af475c,invoice_detection_egvp,done,{'text': '<page_1> Obergerichtsvollzieherin Hü...,"{'answer': '{""is_invoice_inside"": false, ""star...","{'end_page': -1, 'start_page': -1, 'is_invoice...",0,2026-05-29 12:29:38.966440,2026-05-29 12:53:35.371554,65622752,"{'instance': {'private_ip': '100.2.38.122', 'i..."


In [99]:
# only get models for egvp

print(egvp_only)

['pfub_erlass_egvp', 'invoice_detection_egvp', 'vermogenverzeichnis_egvp', 'egvp_standalone_invoice', 'drittauskunft_egvp']


In [101]:
gpu_tasks_egvp = gpu_tasks[gpu_tasks["model_name"].isin(egvp_only)].copy()
gpu_tasks_egvp

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,created_at,updated_at,attachment_id,meta_info
3,f8cdac37-b141-4f0f-ac57-7b255db3c801,3b7fe2c3-62d5-4ff1-8de3-f3c5955c543a,6d4075a2-7bde-5f60-88ac-ea0aed86c059,invoice_detection_egvp,done,{'text': '<page_1> Beleg über die Einlieferung...,"{'answer': '{""is_invoice_inside"": true, ""start...","{'end_page': 1, 'start_page': 1, 'is_invoice_i...",0,2026-05-29 12:29:13.651323,2026-05-29 12:53:31.070652,65612554,"{'instance': {'private_ip': '100.2.38.122', 'i..."
4,378b33de-ead5-42c3-b099-515bba2cc6a4,da1ad76a-a880-4a2d-8b3b-261106498a34,011af6c2-a852-5e94-8039-bdada6af475c,invoice_detection_egvp,done,{'text': '<page_1> Obergerichtsvollzieherin Hü...,"{'answer': '{""is_invoice_inside"": false, ""star...","{'end_page': -1, 'start_page': -1, 'is_invoice...",0,2026-05-29 12:29:38.966440,2026-05-29 12:53:35.371554,65622752,"{'instance': {'private_ip': '100.2.38.122', 'i..."
20,b6d1aeb4-37a9-47d8-901d-d3133a6dc4c4,b1b9c268-f0cb-4b1b-afbf-1349e5aac172,a0b48300-4f85-5cc1-9bde-f60992960acc,invoice_detection_egvp,done,{'text': '<page_1> Amtsgericht Stuttgart Blatt...,"{'answer': '{""is_invoice_inside"": false, ""star...","{'end_page': -1, 'start_page': -1, 'is_invoice...",0,2026-05-22 13:23:44.404244,2026-05-22 14:02:15.139048,65309937,"{'instance': {'private_ip': '100.2.36.225', 'i..."
25,1cbfc513-e51f-4bc2-86f6-a02bf00600f9,32a9125c-9ce7-4af1-acbd-9650b226aacc,82670e5f-3d54-5336-9fc5-76db939a9fae,invoice_detection_egvp,done,{'text': '<page_1> Amtsgericht Wedding Mahnsac...,"{'answer': '{""is_invoice_inside"": true, ""start...","{'end_page': 1, 'start_page': 1, 'is_invoice_i...",0,2026-06-09 10:30:09.121051,2026-06-09 11:06:49.648217,68133940,"{'instance': {'private_ip': '100.2.36.1', 'ins..."
34,57b8d283-aa81-47d5-9ced-ee12e541f5c9,60548fef-e1c9-4593-b9f0-38196ac0a1ba,889db54d-0fc8-5f31-97d9-5d94f92752b3,pfub_erlass_egvp,done,{'text': 'Bundeszentralamt für Steuern POSTANS...,"{'answer': '{""is_pfub"": false, ""is_invoice_ins...","{'is_pfub': False, 'is_invoice_inside': False}",0,2026-05-13 15:25:34.170428,2026-05-13 15:30:47.971993,63143563,"{'instance': {'private_ip': '100.2.36.1', 'ins..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13877,94d855aa-419f-4cea-b89f-7ee740299640,68def108-662e-471c-85c0-535d8141b7fe,e7758b39-7656-506f-b117-64fcf7a9b7f5,drittauskunft_egvp,done,{'text': 'Habig 9 DR II 1917/25 Obergerichtsvo...,"{'answer': '{""is_dritt"":false}', 'answer_reaso...",{'is_dritt': False},0,2026-06-19 19:21:52.765804,2026-06-19 19:22:34.865206,69119648,"{'instance': {'private_ip': '100.2.36.225', 'i..."
13878,93231571-8229-490e-a38d-51e2a70d3117,30546c9d-9f09-47c2-bfca-be7da2b761b0,4eccfffc-5a7e-58ee-88d1-f35bbd4b3a6d,drittauskunft_egvp,done,{'text': 'Geistmarkt 11 MARTINA HENDRICKS 4644...,"{'answer': '{""is_dritt"":false}', 'answer_reaso...",{'is_dritt': False},0,2026-06-19 19:21:51.870420,2026-06-19 19:22:33.834118,69119640,"{'instance': {'private_ip': '100.2.36.225', 'i..."
13879,484b1146-579e-4ab8-8e13-11ff56b64a9d,68def108-662e-471c-85c0-535d8141b7fe,e7758b39-7656-506f-b117-64fcf7a9b7f5,invoice_detection_egvp,done,{'text': '<page_1> Habig 9 DR II 1917/25 Oberg...,"{'answer': '{""is_invoice_inside"": true, ""start...","{'end_page': 1, 'start_page': 1, 'is_invoice_i...",0,2026-06-19 19:21:52.766508,2026-06-19 19:30:56.011591,69119648,"{'instance': {'private_ip': '100.2.36.225', 'i..."
13880,fd51d0a7-2bb8-45f3-a2a6-3fc6c8a70bca,30546c9d-9f09-47c2-bfca-be7da2b761b0,4eccfffc-5a7e-58ee-88d1-f35bbd4b3a6d,pfub_erlass_egvp,done,{'text': 'Geistmarkt 11 MARTINA HENDRICKS 4644...,"{'answer': '{""is_pfub"": false, ""is_invoice_ins...","{'is_pfub': False, 'is_invoice_inside': True}",0,2026-06-19 19:21:51.872252,2026-06-19 19:32:57.663077,69119640,"{'instance': {'private_ip': '100.2.38.65', 'in..."


In [102]:
gpu_tasks_egvp.model_name.value_counts()

model_name
pfub_erlass_egvp            3011
invoice_detection_egvp      1643
drittauskunft_egvp           970
vermogenverzeichnis_egvp     970
egvp_standalone_invoice      593
Name: count, dtype: int64

In [120]:
# merge on ticket_uuid 

egvp_processed_with_gpu = egvp_processed.merge(gpu_tasks_egvp, on="ticket_uuid", how="left", suffixes=("", "_gpu"))

In [121]:

egvp_processed_with_gpu = egvp_processed_with_gpu.drop_duplicates(subset=["ticket_uuid", "attachment_id", "model_name"])

In [133]:
egvp_processed_with_gpu['reasoning_content'] = egvp_processed_with_gpu['model_output'].apply(
    lambda x: x.get('answer_reasoning_content') if isinstance(x, dict) else None
)

In [135]:
egvp_processed_with_gpu['reasoning_content'].isna().sum()

np.int64(0)

In [136]:
egvp_processed_with_gpu['reasoning_content_token_count'] = egvp_processed_with_gpu['reasoning_content'].apply(
    lambda x: len(tokenizer_32B.encode(x, add_special_tokens=False)) if isinstance(x, str) else 0
)

In [141]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

rc_tokens = egvp_processed_with_gpu['reasoning_content_token_count'].replace(0, np.nan).dropna()

sorted_rc = np.sort(rc_tokens)
cdf_rc = np.arange(1, len(sorted_rc) + 1) / len(sorted_rc)

p90 = rc_tokens.quantile(0.90)
p95 = rc_tokens.quantile(0.95)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Reasoning Content Token Count Distribution (n={len(rc_tokens)})",
        "CDF of Reasoning Content Token Count",
    ),
)

fig.add_trace(go.Histogram(
    x=rc_tokens,
    nbinsx=50,
    marker=dict(color="mediumpurple", line=dict(color="black", width=0.5)),
    name="reasoning tokens",
    hovertemplate="tokens: %{x}<br>count: %{y}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=sorted_rc, y=cdf_rc,
    mode="lines",
    line=dict(color="mediumpurple", width=2),
    name="CDF",
    hovertemplate="tokens ≤ %{x}<br>fraction: %{y:.2%}<extra></extra>",
), row=1, col=2)

for col in (1, 2):
    fig.add_vline(
        x=p90, line=dict(color="orange", dash="dash", width=2),
        annotation_text=f"p90 ({p90:.0f})", annotation_position="top right",
        row=1, col=col,
    )
    fig.add_vline(
        x=p95, line=dict(color="firebrick", dash="dash", width=2),
        annotation_text=f"p95 ({p95:.0f})", annotation_position="top right",
        row=1, col=col,
    )

fig.update_xaxes(title_text="Token Count", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_xaxes(title_text="Token Count", row=1, col=2)
fig.update_yaxes(title_text="Cumulative Fraction", row=1, col=2)

fig.update_layout(
    title_text="Reasoning Content Token Count — Processed EGVP (per model output)",
    template="plotly_white",
    showlegend=False,
    height=500, width=1100,
    bargap=0.05,
)
fig.show()

print(rc_tokens.describe().round())
print(f"p90: {p90:.0f}  |  p95: {p95:.0f}")
print(f"\nRows with no reasoning content: {egvp_processed_with_gpu['reasoning_content_token_count'].eq(0).sum()}")
print(f"\nPer model_name:")
print(egvp_processed_with_gpu.groupby('model_name')['reasoning_content_token_count'].describe().round(1))

count    4810.0
mean      524.0
std       172.0
min       176.0
25%       413.0
50%       495.0
75%       595.0
max      2010.0
Name: reasoning_content_token_count, dtype: float64
p90: 712  |  p95: 803

Rows with no reasoning content: 26

Per model_name:
                           count   mean    std    min    25%    50%    75%  \
model_name                                                                   
drittauskunft_egvp         653.0  428.6  133.0  176.0  360.0  413.0  472.0   
egvp_standalone_invoice    421.0  440.6  183.0  234.0  356.0  403.0  468.0   
invoice_detection_egvp    1111.0  648.6  187.1  312.0  531.0  622.0  720.0   
pfub_erlass_egvp          1998.0  537.8  141.5  214.0  454.0  516.0  595.0   
vermogenverzeichnis_egvp   653.0  399.2  117.7    0.0  347.0  400.0  460.0   

                             max  
model_name                        
drittauskunft_egvp        1918.0  
egvp_standalone_invoice   2010.0  
invoice_detection_egvp    1967.0  
pfub_erlass_egvp       

In [146]:
egvp_processed['real_token_count'].min()

np.int64(130)

# Worst-case context window probability

`total_context = prompt_tokens(model) + doc_tokens + reasoning_tokens`

Worst case = max prompt + max doc + max reasoning = `2700 + 6400 + 2000 = 11100`.

Two ways to estimate how likely this is:
1. **Empirical** — build the actual per-request total from the data and measure `P(total >= 11100)`.
2. **Independence assumption** — `P(prompt>=2700) × P(doc>=6400) × P(reasoning>=2000)`, i.e. the chance all three peak at once if they were independent (an upper-bound-ish sanity check; the absolute worst case requires all three maxima to coincide).

In [158]:
import plotly.graph_objects as go
import numpy as np

MAX_PROMPT, MAX_DOC, MAX_REASONING = 2700, 6400, 2000
WORST_CASE = MAX_PROMPT + MAX_DOC + MAX_REASONING  # 11100

# prompt token count is (essentially) fixed per model_name — EGVP models only
prompt_token_map = {m: v["token_count"] for m, v in model_to_token_count_egvp.items()}

ctx = egvp_processed_with_gpu[egvp_processed_with_gpu["model_name"].isin(egvp_only)].copy()
ctx["prompt_token_count"] = ctx["model_name"].map(prompt_token_map)
ctx = ctx.dropna(subset=["prompt_token_count", "real_token_count", "reasoning_content_token_count"])
ctx["total_context"] = (
    ctx["prompt_token_count"] + ctx["real_token_count"] + ctx["reasoning_content_token_count"]
)

# 1) Empirical probability from the actual joint distribution
p_empirical = (ctx["total_context"] >= WORST_CASE).mean()

# 2) Independence assumption: product of marginal tail probabilities
p_prompt = (ctx["prompt_token_count"] >= MAX_PROMPT).mean()
p_doc = (ctx["real_token_count"] >= MAX_DOC).mean()
p_reason = (ctx["reasoning_content_token_count"] >= MAX_REASONING).mean()
p_independent = p_prompt * p_doc * p_reason

print(f"EGVP models only: {egvp_only}")
print(f"n requests = {len(ctx)}")
print("\nMarginal tail probabilities:")
print(f"  P(prompt    >= {MAX_PROMPT}) = {p_prompt:.4f}")
print(f"  P(doc       >= {MAX_DOC}) = {p_doc:.4f}")
print(f"  P(reasoning >= {MAX_REASONING}) = {p_reason:.4f}")
print(f"\nP(all three peak at once, independence)   = {p_independent:.2e}")
print(f"P(total_context >= {WORST_CASE}) empirical = {p_empirical:.4f}")
print(f"\nObserved total_context max = {ctx['total_context'].max():.0f}")
print(ctx["total_context"].describe().round())

# distribution of the actual total context window
sorted_ctx = np.sort(ctx["total_context"])
cdf_ctx = np.arange(1, len(sorted_ctx) + 1) / len(sorted_ctx)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=ctx["total_context"], nbinsx=60,
    marker=dict(color="teal", line=dict(color="black", width=0.5)),
    name="total_context",
    hovertemplate="tokens: %{x}<br>count: %{y}<extra></extra>",
))
for q, c in [(0.90, "orange"), (0.95, "darkorange"), (0.99, "firebrick")]:
    qv = ctx["total_context"].quantile(q)
    fig.add_vline(x=qv, line=dict(color=c, dash="dot", width=1.5),
                  annotation_text=f"p{int(q*100)} ({qv:.0f})", annotation_position="top")
fig.add_vline(x=WORST_CASE, line=dict(color="red", dash="dash", width=2),
              annotation_text=f"Worst case ({WORST_CASE})", annotation_position="top left")
fig.update_layout(
    title_text="Actual Total Context Window — prompt + doc + reasoning (EGVP models only)",
    xaxis_title="total_context (tokens)", yaxis_title="Count",
    template="plotly_white", height=500, width=1000, bargap=0.03,
)
fig.show()

EGVP models only: ['pfub_erlass_egvp', 'invoice_detection_egvp', 'vermogenverzeichnis_egvp', 'egvp_standalone_invoice', 'drittauskunft_egvp']
n requests = 4836

Marginal tail probabilities:
  P(prompt    >= 2700) = 0.0000
  P(doc       >= 6400) = 0.0000
  P(reasoning >= 2000) = 0.0002

P(all three peak at once, independence)   = 0.00e+00
P(total_context >= 11100) empirical = 0.0000

Observed total_context max = 9632
count    4836.0
mean     3531.0
std      1026.0
min      1939.0
25%      2745.0
50%      3286.0
75%      4059.0
max      9632.0
Name: total_context, dtype: float64


In [159]:
# Empirical context-window sizing: pick a high percentile of the ACTUAL total_context
# instead of the theoretical worst case (11,100), and report headroom vs that worst case.

percentiles = [0.95, 0.99, 0.999, 1.0]
labels = {0.95: "p95", 0.99: "p99", 0.999: "p99.9", 1.0: "max"}

print(f"Theoretical worst case = {WORST_CASE} tokens  |  n requests = {len(ctx)}\n")
print(f"{'cutoff':>8} {'tokens':>8} {'coverage':>10} {'headroom vs 11100':>20}")
sizing = []
for q in percentiles:
    cutoff = ctx["total_context"].quantile(q)
    coverage = (ctx["total_context"] <= cutoff).mean()  # fraction served without truncation
    headroom = WORST_CASE - cutoff                       # spare tokens vs worst case
    headroom_pct = headroom / WORST_CASE * 100
    sizing.append({"cutoff": labels[q], "tokens": round(cutoff), "coverage": coverage,
                   "headroom_tokens": round(headroom), "headroom_pct": round(headroom_pct, 1)})
    print(f"{labels[q]:>8} {cutoff:>8.0f} {coverage:>9.2%} {headroom:>13.0f} ({headroom_pct:>4.1f}%)")

sizing_df = pd.DataFrame(sizing)

# Recommendation: size to p99 with a small safety margin, well below the worst case.
p99_cutoff = ctx["total_context"].quantile(0.99)
recommended = int(np.ceil(p99_cutoff / 100) * 100)  # round up to nearest 100 for a clean budget
print(f"\nRecommended context window (p99 rounded up) = {recommended} tokens")
print(f"  -> serves {(ctx['total_context'] <= recommended).mean():.2%} of requests")
print(f"  -> {WORST_CASE - recommended} tokens ({(WORST_CASE - recommended)/WORST_CASE*100:.1f}%) smaller than the {WORST_CASE} worst case")

sizing_df

Theoretical worst case = 11100 tokens  |  n requests = 4836

  cutoff   tokens   coverage    headroom vs 11100
     p95     5628    95.00%          5472 (49.3%)
     p99     7005    98.99%          4095 (36.9%)
   p99.9     8325    99.90%          2775 (25.0%)
     max     9632   100.00%          1468 (13.2%)

Recommended context window (p99 rounded up) = 7100 tokens
  -> serves 99.15% of requests
  -> 4000 tokens (36.0%) smaller than the 11100 worst case


,cutoff,tokens,coverage,headroom_tokens,headroom_pct
0,p95,5628,0.949959,5472,49.3
1,p99,7005,0.989868,4095,36.9
2,p99.9,8325,0.998966,2775,25.0
3,max,9632,1.000000,1468,13.2


In [160]:
# p99 of each context component (EGVP requests only)
components = {
    "EGVP prompt length":        ctx["prompt_token_count"],
    "EGVP input document length": ctx["real_token_count"],
    "Reasoning content length":   ctx["reasoning_content_token_count"],
}

print(f"n requests = {len(ctx)}\n")
print(f"{'component':>28} {'p99':>8} {'max':>8} {'mean':>8}")
p99_rows = []
for name, series in components.items():
    p99 = series.quantile(0.99)
    p99_rows.append({"component": name, "p99": round(p99), "max": round(series.max()), "mean": round(series.mean())})
    print(f"{name:>28} {p99:>8.0f} {series.max():>8.0f} {series.mean():>8.0f}")

print(f"\nSum of individual p99s = {sum(r['p99'] for r in p99_rows):.0f} tokens")
pd.DataFrame(p99_rows)

n requests = 4836

                   component      p99      max     mean
          EGVP prompt length     2667     2667     1851
  EGVP input document length     4074     6377     1159
    Reasoning content length     1156     2010      521

Sum of individual p99s = 7897 tokens


,component,p99,max,mean
0,EGVP prompt length,2667,2667,1851
1,EGVP input document length,4074,6377,1159
2,Reasoning content length,1156,2010,521


In [163]:
# Probability that all THREE components hit their p99 simultaneously
# (i.e. the "sum of p99s = 7897" worst case actually occurring)

p99_prompt = ctx["prompt_token_count"].quantile(0.99)
p99_doc = ctx["real_token_count"].quantile(0.99)
p99_reason = ctx["reasoning_content_token_count"].quantile(0.99)

# marginal tail probabilities at each p99 (≈ 0.01 by definition, but compute from data)
m_prompt = (ctx["prompt_token_count"] >= p99_prompt).mean()
m_doc = (ctx["real_token_count"] >= p99_doc).mean()
m_reason = (ctx["reasoning_content_token_count"] >= p99_reason).mean()

# 1) Independence assumption: multiply the marginals
p_indep = m_prompt * m_doc * m_reason

# 2) Empirical: fraction of requests where all three are >= their own p99 at the same time
all_three = (
    (ctx["prompt_token_count"] >= p99_prompt)
    & (ctx["real_token_count"] >= p99_doc)
    & (ctx["reasoning_content_token_count"] >= p99_reason)
)
p_empirical_all = all_three.mean()

print(f"n requests = {len(ctx)}\n")
print(f"Marginal P(component >= its p99):")
print(f"  prompt    >= {p99_prompt:.0f} : {m_prompt:.4f}")
print(f"  doc       >= {p99_doc:.0f} : {m_doc:.4f}")
print(f"  reasoning >= {p99_reason:.0f} : {m_reason:.4f}")
print(f"\nP(all three >= p99 together):")
print(f"  independence assumption = {p_indep:.2e}  (~0.01^3 = 1e-6)")
print(f"  empirical (from data)   = {p_empirical_all:.2e}  ({all_three.sum()} of {len(ctx)} requests)")
print(f"\nExpected once per ~{1/p_indep:,.0f} requests (independence) "
      f"vs ~{(1/p_empirical_all) if p_empirical_all else float('inf'):,.0f} (empirical)")

n requests = 4836

Marginal P(component >= its p99):
  prompt    >= 2667 : 0.2297
  doc       >= 4074 : 0.0103
  reasoning >= 1156 : 0.0101

P(all three >= p99 together):
  independence assumption = 2.41e-05  (~0.01^3 = 1e-6)
  empirical (from data)   = 2.07e-04  (1 of 4836 requests)

Expected once per ~41,551 requests (independence) vs ~4,836 (empirical)


In [169]:
# ============================================================
# What happens if we raise the attachment limit 1500 -> 2100 ?
#   - extra EGVP documents (simple_token_count in (1500, 2100]) now get accepted
#   - those new docs were rejected before -> NO gpu task -> reuse CURRENT reasoning stats
#   - prompt tokens are fixed per model -> UNCHANGED
#   - only the INPUT DOCUMENT size distribution grows
# ============================================================
OLD_LIMIT, NEW_LIMIT = 1500, 2100

docs = egvp_sampled_one_month.dropna(subset=["simple_token_count", "real_token_count"]).copy()

old_doc = docs.loc[docs["simple_token_count"] <= OLD_LIMIT, "real_token_count"]
new_doc = docs.loc[docs["simple_token_count"] <= NEW_LIMIT, "real_token_count"]
added   = docs.loc[(docs["simple_token_count"] > OLD_LIMIT)
                   & (docs["simple_token_count"] <= NEW_LIMIT), "real_token_count"]

n_old, n_new, n_added = len(old_doc), len(new_doc), len(added)
print("=== Accepted EGVP attachments (one-month sample) ===")
print(f"limit {OLD_LIMIT}: {n_old} docs accepted")
print(f"limit {NEW_LIMIT}: {n_new} docs accepted  (+{n_added} docs, +{n_added / n_old * 100:.1f}%)")
print(f"newly accepted are {n_added / n_new * 100:.1f}% of the new accepted set\n")

# --- Input document size component (real / Qwen tokens) ---
def doc_stats(s):
    return dict(mean=s.mean(), p95=s.quantile(0.95), p99=s.quantile(0.99), max=s.max())

so, sn = doc_stats(old_doc), doc_stats(new_doc)
print("=== Input document size (real tokens) ===")
print(f"{'limit':>6}{'mean':>10}{'p95':>10}{'p99':>10}{'max':>10}")
print(f"{OLD_LIMIT:>6}{so['mean']:>10.0f}{so['p95']:>10.0f}{so['p99']:>10.0f}{so['max']:>10.0f}")
print(f"{NEW_LIMIT:>6}{sn['mean']:>10.0f}{sn['p95']:>10.0f}{sn['p99']:>10.0f}{sn['max']:>10.0f}")
print(f"{'delta':>6}{sn['mean'] - so['mean']:>10.0f}{sn['p95'] - so['p95']:>10.0f}"
      f"{sn['p99'] - so['p99']:>10.0f}{sn['max'] - so['max']:>10.0f}\n")

# --- Prompt & reasoning held fixed from CURRENT data ---
prompt = ctx["prompt_token_count"]
reason = ctx["reasoning_content_token_count"]
p99_prompt, max_prompt = prompt.quantile(0.99), prompt.max()
p99_reason, max_reason = reason.quantile(0.99), reason.max()

# --- Resulting context window (prompt fixed + doc grows + reasoning fixed) ---
def window(doc_p99, doc_max):
    return dict(p99_sum=p99_prompt + doc_p99 + p99_reason,
                worst=max_prompt + doc_max + max_reason)

w_old, w_new = window(so["p99"], so["max"]), window(sn["p99"], sn["max"])
print("=== Resulting total context window ===")
print(f"{'limit':>6}{'p99-sum':>12}{'worst-case':>14}")
print(f"{OLD_LIMIT:>6}{w_old['p99_sum']:>12.0f}{w_old['worst']:>14.0f}")
print(f"{NEW_LIMIT:>6}{w_new['p99_sum']:>12.0f}{w_new['worst']:>14.0f}")
print(f"{'delta':>6}{w_new['p99_sum'] - w_old['p99_sum']:>12.0f}"
      f"{w_new['worst'] - w_old['worst']:>14.0f}\n")

# --- Joint total_context p99 via independent resampling ---
# new docs lack per-request reasoning, so combine doc draws with CURRENT prompt+reasoning draws
rng = np.random.default_rng(0)
N = 200_000
sim_old = rng.choice(prompt.values, N) + rng.choice(old_doc.values, N) + rng.choice(reason.values, N)
sim_new = rng.choice(prompt.values, N) + rng.choice(new_doc.values, N) + rng.choice(reason.values, N)
print("=== Joint total_context (independent resampling) ===")
print(f"{'limit':>6}{'p99':>10}{'p99.9':>10}{'max':>10}")
print(f"{OLD_LIMIT:>6}{np.quantile(sim_old, 0.99):>10.0f}{np.quantile(sim_old, 0.999):>10.0f}{sim_old.max():>10.0f}")
print(f"{NEW_LIMIT:>6}{np.quantile(sim_new, 0.99):>10.0f}{np.quantile(sim_new, 0.999):>10.0f}{sim_new.max():>10.0f}")
for w in (8192,):
    print(f"\nFits in {w} tokens: limit {OLD_LIMIT} -> {(sim_old <= w).mean():.4%}, "
          f"limit {NEW_LIMIT} -> {(sim_new <= w).mean():.4%}")


=== Accepted EGVP attachments (one-month sample) ===
limit 1500: 1999 docs accepted
limit 2100: 2361 docs accepted  (+362 docs, +18.1%)
newly accepted are 15.3% of the new accepted set

=== Input document size (real tokens) ===
 limit      mean       p95       p99       max
  1500      1195      3113      4218      6377
  2100      1819      5560      6217      9259
 delta       625      2447      1999      2882

=== Resulting total context window ===
 limit     p99-sum    worst-case
  1500        8041         11054
  2100       10040         13936
 delta        1999          2882

=== Joint total_context (independent resampling) ===
 limit       p99     p99.9       max
  1500      6923      8156     10036
  2100      9015     10984     13501

Fits in 8192 tokens: limit 1500 -> 99.9030%, limit 2100 -> 96.1225%


In [170]:
# ============================================================
# What happens if we raise the attachment limit 1500 -> 2100 ?
#   - extra EGVP documents (simple_token_count in (1500, 2100]) now get accepted
#   - those new docs were rejected before -> NO gpu task -> reuse CURRENT reasoning stats
#   - prompt tokens are fixed per model -> UNCHANGED
#   - only the INPUT DOCUMENT size distribution grows
# ============================================================
OLD_LIMIT, NEW_LIMIT = 1500, 2000

docs = egvp_sampled_one_month.dropna(subset=["simple_token_count", "real_token_count"]).copy()

old_doc = docs.loc[docs["simple_token_count"] <= OLD_LIMIT, "real_token_count"]
new_doc = docs.loc[docs["simple_token_count"] <= NEW_LIMIT, "real_token_count"]
added   = docs.loc[(docs["simple_token_count"] > OLD_LIMIT)
                   & (docs["simple_token_count"] <= NEW_LIMIT), "real_token_count"]

n_old, n_new, n_added = len(old_doc), len(new_doc), len(added)
print("=== Accepted EGVP attachments (one-month sample) ===")
print(f"limit {OLD_LIMIT}: {n_old} docs accepted")
print(f"limit {NEW_LIMIT}: {n_new} docs accepted  (+{n_added} docs, +{n_added / n_old * 100:.1f}%)")
print(f"newly accepted are {n_added / n_new * 100:.1f}% of the new accepted set\n")

# --- Input document size component (real / Qwen tokens) ---
def doc_stats(s):
    return dict(mean=s.mean(), p95=s.quantile(0.95), p99=s.quantile(0.99), max=s.max())

so, sn = doc_stats(old_doc), doc_stats(new_doc)
print("=== Input document size (real tokens) ===")
print(f"{'limit':>6}{'mean':>10}{'p95':>10}{'p99':>10}{'max':>10}")
print(f"{OLD_LIMIT:>6}{so['mean']:>10.0f}{so['p95']:>10.0f}{so['p99']:>10.0f}{so['max']:>10.0f}")
print(f"{NEW_LIMIT:>6}{sn['mean']:>10.0f}{sn['p95']:>10.0f}{sn['p99']:>10.0f}{sn['max']:>10.0f}")
print(f"{'delta':>6}{sn['mean'] - so['mean']:>10.0f}{sn['p95'] - so['p95']:>10.0f}"
      f"{sn['p99'] - so['p99']:>10.0f}{sn['max'] - so['max']:>10.0f}\n")

# --- Prompt & reasoning held fixed from CURRENT data ---
prompt = ctx["prompt_token_count"]
reason = ctx["reasoning_content_token_count"]
p99_prompt, max_prompt = prompt.quantile(0.99), prompt.max()
p99_reason, max_reason = reason.quantile(0.99), reason.max()

# --- Resulting context window (prompt fixed + doc grows + reasoning fixed) ---
def window(doc_p99, doc_max):
    return dict(p99_sum=p99_prompt + doc_p99 + p99_reason,
                worst=max_prompt + doc_max + max_reason)

w_old, w_new = window(so["p99"], so["max"]), window(sn["p99"], sn["max"])
print("=== Resulting total context window ===")
print(f"{'limit':>6}{'p99-sum':>12}{'worst-case':>14}")
print(f"{OLD_LIMIT:>6}{w_old['p99_sum']:>12.0f}{w_old['worst']:>14.0f}")
print(f"{NEW_LIMIT:>6}{w_new['p99_sum']:>12.0f}{w_new['worst']:>14.0f}")
print(f"{'delta':>6}{w_new['p99_sum'] - w_old['p99_sum']:>12.0f}"
      f"{w_new['worst'] - w_old['worst']:>14.0f}\n")

# --- Joint total_context p99 via independent resampling ---
# new docs lack per-request reasoning, so combine doc draws with CURRENT prompt+reasoning draws
rng = np.random.default_rng(0)
N = 200_000
sim_old = rng.choice(prompt.values, N) + rng.choice(old_doc.values, N) + rng.choice(reason.values, N)
sim_new = rng.choice(prompt.values, N) + rng.choice(new_doc.values, N) + rng.choice(reason.values, N)
print("=== Joint total_context (independent resampling) ===")
print(f"{'limit':>6}{'p99':>10}{'p99.9':>10}{'max':>10}")
print(f"{OLD_LIMIT:>6}{np.quantile(sim_old, 0.99):>10.0f}{np.quantile(sim_old, 0.999):>10.0f}{sim_old.max():>10.0f}")
print(f"{NEW_LIMIT:>6}{np.quantile(sim_new, 0.99):>10.0f}{np.quantile(sim_new, 0.999):>10.0f}{sim_new.max():>10.0f}")
for w in (8192,):
    print(f"\nFits in {w} tokens: limit {OLD_LIMIT} -> {(sim_old <= w).mean():.4%}, "
          f"limit {NEW_LIMIT} -> {(sim_new <= w).mean():.4%}")


=== Accepted EGVP attachments (one-month sample) ===
limit 1500: 1999 docs accepted
limit 2000: 2304 docs accepted  (+305 docs, +15.3%)
newly accepted are 13.2% of the new accepted set

=== Input document size (real tokens) ===
 limit      mean       p95       p99       max
  1500      1195      3113      4218      6377
  2000      1721      5310      6123      9259
 delta       526      2197      1905      2882

=== Resulting total context window ===
 limit     p99-sum    worst-case
  1500        8041         11054
  2000        9946         13936
 delta        1905          2882

=== Joint total_context (independent resampling) ===
 limit       p99     p99.9       max
  1500      6923      8156     10036
  2000      8859     11055     13144

Fits in 8192 tokens: limit 1500 -> 99.9030%, limit 2000 -> 97.0450%


In [171]:
# ============================================================
# Pick the new text-length limit: sweep candidate limits and check, for each,
#   - how many more docs get accepted (recall gain)
#   - the joint total_context p99 / p99.9 (prompt + doc + reasoning, resampled)
#   - whether it still fits an 8k (8192) context window
# Reasoning + prompt are held fixed (current data); only the document set grows.
# ============================================================
WINDOW = 8192  # target serving context window

prompt = ctx["prompt_token_count"].values
reason = ctx["reasoning_content_token_count"].values
rng = np.random.default_rng(0)
N = 200_000
prompt_draw = rng.choice(prompt, N)
reason_draw = rng.choice(reason, N)

docs = egvp_sampled_one_month.dropna(subset=["simple_token_count", "real_token_count"])
base = (docs["simple_token_count"] <= 1500).sum()  # current accepted count

rows = []
for lim in range(1500, 2601, 100):
    accepted_doc = docs.loc[docs["simple_token_count"] <= lim, "real_token_count"].values
    sim = prompt_draw + rng.choice(accepted_doc, N) + reason_draw
    rows.append({
        "limit": lim,
        "n_accepted": len(accepted_doc),
        "recall_gain_%": (len(accepted_doc) - base) / base * 100,
        "doc_p99": np.quantile(accepted_doc, 0.99),
        "joint_p99": np.quantile(sim, 0.99),
        "joint_p99.9": np.quantile(sim, 0.999),
        f"fit_{WINDOW}_%": (sim <= WINDOW).mean() * 100,
    })

sweep = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}")
print(sweep.to_string(index=False))

# largest limit that keeps the joint p99 within the 8k window
ok = sweep[sweep["joint_p99"] <= WINDOW]
best = int(ok["limit"].max()) if len(ok) else None
print(f"\nLargest limit with joint p99 <= {WINDOW}: {best}")
ok999 = sweep[sweep["joint_p99.9"] <= WINDOW]
best999 = int(ok999["limit"].max()) if len(ok999) else None
print(f"Largest limit with joint p99.9 <= {WINDOW} (stricter): {best999}")
pd.reset_option("display.float_format")


 limit  n_accepted  recall_gain_%  doc_p99  joint_p99  joint_p99.9  fit_8192_%
  1500        1999              0    4,218      6,939        8,118         100
  1600        2093              5    5,038      7,701        9,066         100
  1700        2155              8    5,399      8,102        9,613          99
  1800        2206             10    5,700      8,400       10,004          99
  1900        2264             13    6,027      8,663       11,049          98
  2000        2304             15    6,123      8,874       11,027          97
  2100        2361             18    6,217      9,015       11,104          96
  2200        2447             22    6,387      9,193       11,198          95
  2300        2482             24    6,783      9,413       11,083          94
  2400        2533             27    6,965      9,737       11,303          92
  2500        2640             32    7,135     10,045       11,775          88
  2600        2751             38    7,394     10,29

# downlaod some docs with new limit <= 2100

In [177]:
added_full = docs.loc[(docs["simple_token_count"] > OLD_LIMIT)
                   & (docs["simple_token_count"] <= NEW_LIMIT)]


In [178]:
added_full

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,s3_key,s3_bucket,job_id,textract_status,s3_link,text,simple_token_count,real_token_count
2,3519795,38d02160-d8c8-5f77-bc14-a519b9c6464a,61609291,NaN,NaN,rejected_too_long,61609291_Dokumente_42326_30042026_143651.pdf,pdf,2026-05-01 12:12:02,2026-05-01 11:44:00,ocr_source_files/2026-05-01/egvp_id_359530/616...,pair-data-engineering-new,e81857c58d61a105e970e7ad8ff5af509ef46c18f86b78...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Sandra Kaiser\nKirchstraße 9 15\nObergerichtsv...,1888,5413
23,3523570,6b22b9d5-a85b-5b0d-960e-44c8b3ca9c55,61920022,NaN,NaN,rejected_too_long,61920022_VV325_221225_583_Abschrift.pdf,pdf,2026-05-03 16:23:49,2026-05-03 15:44:00,ocr_source_files/2026-05-03/egvp_id_359636/619...,pair-data-engineering-new,c75f8ce40ae17a59761ed83d65ac26a23b95d85212cb18...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d.:\nGVin Nadine Berk...,1514,4314
41,3528333,8fea4c67-571a-5112-9141-e4894c73e455,61970816,NaN,NaN,rejected_too_long,61970816_Dokumente_31026_04052026_122055_signe...,pdf,2026-05-04 12:30:44,2026-05-04 11:44:00,ocr_source_files/2026-05-04/egvp_id_360531/619...,pair-data-engineering-new,37a29ed742c902c1e78503641646ec8505813c6d6e981b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,S. Graffenberger\nSegelfliegerdamm 89\nGericht...,1752,4975
43,3528885,aa06250f-ce6d-55f2-a49e-8976dca75bf4,61973128,NaN,NaN,rejected_too_long,61973128_1_Sammel1.pdf,pdf,2026-05-04 13:30:03,2026-05-04 12:44:00,ocr_source_files/2026-05-04/egvp_id_360851/619...,pair-data-engineering-new,c5663bff37a546bd72a0110cde5dd2b9f90baa2a224f4d...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieherin\nAlleestraße 4\nD. Po...,1682,5344
44,3528901,3be5e6b4-aabc-5e19-a8f1-4a8e67d3c4ae,61973335,NaN,NaN,rejected_too_long,61973335_Dokumente_75226_04052026_113006_signe...,pdf,2026-05-04 13:30:03,2026-05-04 12:44:00,ocr_source_files/2026-05-04/egvp_id_360895/619...,pair-data-engineering-new,c0003f18a17c0ce7dbff95ecc042b18513eec293051829...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,J. Murrweiß\nSpeyerer Straße 13\nGerichtsvollz...,1977,5659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1503,3741677,f2976ede-480b-5b6a-8cd1-ce5f5d43f3f1,69110105,NaN,NaN,rejected_too_long,69110105_Vermoegensverzeichnis_zu_DRII-053726.pdf,pdf,2026-06-19 09:23:35,2026-06-19 08:44:00,ocr_source_files/2026-06-19/egvp_id_392450/691...,pair-data-engineering-new,deed585f14484a7987f28958a99892b089e7a7d878c7af...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d. Obergerichtsvollzi...,1636,4791
1505,3742156,f3f3d764-6669-53ff-a2b4-f7806d59a415,69110955,NaN,NaN,rejected_too_long,69110955_Dokumente_60726_19062026_104312.pdf,pdf,2026-06-19 10:23:37,2026-06-19 09:44:00,ocr_source_files/2026-06-19/egvp_id_392506/691...,pair-data-engineering-new,86287d06516b2177a2a352dd4fbaac95ac501ef418520c...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Christian Franke\nRatsstraße 17\nGerichtsvollz...,1829,5412
1516,3743776,36034129-ef4b-5d31-b96d-2d4a33a81a7e,69114495,NaN,NaN,rejected_too_long,69114495_1_Sammel1.pdf,pdf,2026-06-19 13:29:27,2026-06-19 12:44:00,ocr_source_files/2026-06-19/egvp_id_392682/691...,pair-data-engineering-new,772aa0b3a3aa1dbc389595682baa72dcade9d3156e890e...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anita Jakob\n93049 Regensburg\nObergerichtsvol...,1785,5913
1524,3745044,a4766011-f81c-5121-a449-bf8ee7bc9aaf,69120056,NaN,NaN,rejected_too_long,69120056_VV_PossmannSigwarthJana_signed.pdf,pdf,2026-06-19 21:12:24,2026-06-19 20:44:00,ocr_source_files/2026-06-19/egvp_id_392793/691...,pair-data-engineering-new,e9bdbd5f07fdef962bbed22e05a937af1e4fea0df28b93...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d.:\nGV' in Marie-Kri..

In [180]:
added_full.attachment_id.nunique()

305

In [181]:
downlaod_added_sample = added_full.sample(n=100, random_state=0)


In [182]:
downlaod_added_sample

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,s3_key,s3_bucket,job_id,textract_status,s3_link,text,simple_token_count,real_token_count
272,3557286,950963d2-25ef-55a4-9168-a973ab94c89d,62496631,NaN,NaN,rejected_too_long,62496631_0_Sammel1.pdf,pdf,2026-05-10 06:14:15,2026-05-10 05:44:00,ocr_source_files/2026-05-10/egvp_id_365875/624...,pair-data-engineering-new,29db8fb3a487c593ef62780da113d1f2cb98aebcfdfdfc...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Verhülsdonk\n24306 Plön\nObergerichtsvollziehe...,1998,5834
243,3552551,b2dae5f4-ba62-558c-988b-e4b602e8c44e,62342889,NaN,NaN,rejected_too_long,62342889_Vermoegensverzeichnis_zu_DRII-042126.pdf,pdf,2026-05-08 10:32:43,2026-05-08 09:44:00,ocr_source_files/2026-05-08/egvp_id_365378/623...,pair-data-engineering-new,c479f941a0c2c58b38ee8ee5d62badf0bb411e521abeee...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d. Obergerichtsvollzi...,1680,4886
267,3555597,c975da8d-a669-5f1d-a607-1af9750cc0e6,62469401,NaN,NaN,rejected_too_long,62469401_Dokumente_62926_09052026_083853-signe...,pdf,2026-05-09 08:14:21,2026-05-09 07:44:00,ocr_source_files/2026-05-09/egvp_id_365827/624...,pair-data-engineering-new,9b5965a5788efc6afb33a324f39f6afcfb7737d53c571e...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Thomas Stauss\nHardtgasse 6\nObergerichtsvollz...,1948,5671
681,3605578,69b1ab20-d395-5f7b-bc81-7fed515c329e,65188395,NaN,NaN,rejected_too_long,65188395_0_Sammel1.pdf,pdf,2026-05-20 15:39:27,2026-05-20 14:44:00,ocr_source_files/2026-05-20/egvp_id_373707/651...,pair-data-engineering-new,253371ad705f6ac6e14db2ddb6bb7bbb8b800f406f3465...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieherin Claudia Janßen\nAmtsg...,1598,5214
1503,3741677,f2976ede-480b-5b6a-8cd1-ce5f5d43f3f1,69110105,NaN,NaN,rejected_too_long,69110105_Vermoegensverzeichnis_zu_DRII-053726.pdf,pdf,2026-06-19 09:23:35,2026-06-19 08:44:00,ocr_source_files/2026-06-19/egvp_id_392450/691...,pair-data-engineering-new,deed585f14484a7987f28958a99892b089e7a7d878c7af...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d. Obergerichtsvollzi...,1636,4791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354,3567536,63c88b75-fd5f-58cb-ae30-7bc644fce40e,63098321,NaN,NaN,rejected_too_long,63098321_DR_II_254_26_aus_Schreibmaschine.PDF,pdf,2026-05-12 09:28:45,2026-05-12 08:44:00,ocr_source_files/2026-05-12/egvp_id_367882/630...,pair-data-engineering-new,d2569e50004519296d6190e1323880bc33e087cb016065...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieher Thomas Köhler\nEichhahn...,1533,4606
752,3615460,0e4dfa2d-02bf-5d70-b6f4-e734698b5e26,65310018,NaN,NaN,rejected_too_long,65310018_VV_993251-signed.pdf,pdf,2026-05-22 13:22:49,2026-05-22 12:44:00,ocr_source_files/2026-05-22/egvp_id_376113/653...,pair-data-engineering-new,2b4f27fc2325e7ac66b9f46a4464a6b7b9b028015792c8...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d.:\nOGV Armin Ferber...,1523,4278
1335,3711918,4c7f94ff-9edd-5162-9c5a-c8cc786d7551,68336237,NaN,NaN,rejected_too_long,68336237_Vermoegensverzeichnis_zu_DRII-167925-...,pdf,2026-06-12 15:18:39,2026-06-12 14:44:00,ocr_source_files/2026-06-12/egvp_id_387275/683...,pair-data-engineering-new,36721bca09cab895f9223119f9c23b2bb5a258d8a1c451...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Anlage zur Niederschrift d. Obergerichtsvollzi...,1606,4625
1157,3683508,72218c68-30b7-5cd8-8409-9c2b3682ab21,67892699,NaN,NaN,rejected_too_long,67892699_0_Sammel1.pdf,pdf,2026-06-08 10:26:00,2026-06-08 09:44:00,ocr_source_files/2026-06-08/egvp_id_383417/678...,pair-data-engineering-new,3cd8ef6e6e6220e8eef62666cc6e0a9c11b4bfe3c62158...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Gerichtsvollzieher\nIndustriestr. 20\nDennis L...,1543,4836


In [187]:
import boto3

session = boto3.Session(
    region_name="eu-central-1",
    profile_name="739275445236_DataScienceUser",
)
s3_client = session.client("s3")

PDF_DOWNLOAD_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp_added_rejected_docs"
os.makedirs(PDF_DOWNLOAD_DIR, exist_ok=True)

for _, row in downlaod_added_sample.iterrows():
    attachment_id = row["attachment_id"]
    bucket = row["s3_bucket"]
    key = row["s3_key"]
    try:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        pdf_path = os.path.join(PDF_DOWNLOAD_DIR, f"{attachment_id}.pdf")
        with open(pdf_path, "wb") as f:
            f.write(response["Body"].read())
    except Exception as e:
        print(f"Failed for {attachment_id}: {e}")

print(f"Done. PDFs saved to {PDF_DOWNLOAD_DIR}")


INFO [2026-06-23 16:48:28] - Found credentials in shared credentials file: ~/.aws/credentials


Done. PDFs saved to /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp_added_rejected_docs
